[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/duckdb-certified/notebooks/day-09-persistent-db-transactions.ipynb#scrollTo=aabb1122)

---
# Day 9 · Persistent Databases, Transactions, and ACID Guarantees
**certified-journeys / duckdb-certified** &nbsp;|&nbsp; Persistence & Transactions

> **Goal for today:** Connect to persistent DuckDB files, demonstrate ACID behaviour with COMMIT and ROLLBACK, use ATTACH to query across multiple databases, and implement a safe idempotent upsert pattern.

---
## DuckDB persistence model

DuckDB operates in two modes:

| Mode | Connection call | Use when |
|---|---|---|
| **In-memory** | `duckdb.connect()` | Ephemeral — data lost on close |
| **Persistent** | `duckdb.connect('analytics.duckdb')` | Data survives process restarts |

The persistent file is a single `.duckdb` binary file. You can `ATTACH` additional files to join across databases.

> **Read first:** [DuckDB connection overview](https://duckdb.org/docs/connect/overview)

---
## ACID guarantees in DuckDB

| Property | How DuckDB implements it |
|---|---|
| **Atomicity** | Transactions either commit fully or roll back completely |
| **Consistency** | Type constraints and NOT NULL enforced at commit |
| **Isolation** | Snapshot isolation — readers see a consistent snapshot |
| **Durability** | Committed data is fsynced to the `.duckdb` file |

In [ ]:
%pip install -q duckdb

---
## Step 1 · Create a persistent DuckDB file and verify it on disk

When you call `duckdb.connect('path/to/file.duckdb')`, DuckDB creates the file on first open and flushes writes to it on every commit.

In [ ]:
import duckdb, os, pathlib

db_path = '/tmp/analytics.duckdb'
# Remove any leftover from a prior run
for p in [db_path, db_path + '.wal']:
    if os.path.exists(p):
        os.remove(p)

# Open a persistent database — file is created if it does not exist
con = duckdb.connect(db_path)

# Verify the file exists on disk immediately after connect
file_exists = os.path.exists(db_path)
print(f"File exists: {file_exists}")
print(f"File path:   {db_path}")
print(f"File size after connect (before any writes): {os.path.getsize(db_path):,} bytes")

# Create a table and insert some rows
con.execute("""
CREATE TABLE events (
    event_id   INTEGER PRIMARY KEY,
    event_type VARCHAR NOT NULL,
    payload    JSON,
    created_at TIMESTAMP DEFAULT current_timestamp
)
""")

con.execute("""
INSERT INTO events (event_id, event_type, payload) VALUES
    (1, 'page_view',  '{"url": "/home", "user": 42}'),
    (2, 'click',      '{"element": "cta_button", "user": 42}'),
    (3, 'purchase',   '{"product_id": 7, "amount": 49.99, "user": 99}')
""")

print(f"\nFile size after write: {os.path.getsize(db_path):,} bytes")
print("\nRows in events:")
print(con.execute("SELECT event_id, event_type, created_at FROM events").df().to_string(index=False))

**What just happened?**
- `duckdb.connect('path')` created a real file — not just an in-memory structure
- The file grew after `CREATE TABLE` + `INSERT` — data was written to disk
- `PRIMARY KEY` and `NOT NULL` constraints are stored in the schema and enforced at INSERT time
- `DEFAULT current_timestamp` sets the value automatically without specifying it in INSERT

---
## Step 2 · Verify data persists across connections

The whole point of a persistent file: close the connection, reopen it, data is still there.

In [ ]:
# Close the current connection
con.close()
print("Connection closed.")

# Re-open the same file — data should still be there
con2 = duckdb.connect(db_path)
rows = con2.execute("SELECT count(*) FROM events").fetchone()[0]
print(f"Rows after reopening: {rows}  (expected: 3)")

# Reuse con2 for the rest of the notebook
con = con2

**What just happened?**
- `con.close()` flushed all pending writes and released the file lock
- Re-opening the file returns all previously committed data — this is **durability**
- The file is fully self-contained — no server process needed between sessions

---
## Step 3 · Transactions — BEGIN, COMMIT, and ROLLBACK

DuckDB auto-commits every statement by default. For multi-statement atomic operations, wrap them in an explicit transaction.

```sql
BEGIN;
  INSERT ...;
  UPDATE ...;
COMMIT;  -- or ROLLBACK to undo everything
```

> **Reference:** [DuckDB transactions](https://duckdb.org/docs/sql/statements/transactions)

In [ ]:
# Demonstrate ROLLBACK: insert rows, then roll back

print("Before transaction:", con.execute("SELECT count(*) FROM events").fetchone()[0], "rows")

# Start a transaction
con.execute("BEGIN")

# Insert two rows inside the transaction
con.execute("""
INSERT INTO events (event_id, event_type, payload) VALUES
    (100, 'signup',  '{"user": 200, "plan": "trial"}'),
    (101, 'logout',  '{"user": 200}')
""")

# Rows ARE visible within the same transaction before commit
mid_count = con.execute("SELECT count(*) FROM events").fetchone()[0]
print(f"During transaction (before commit): {mid_count} rows")

# Simulate an error: roll back instead of committing
con.execute("ROLLBACK")

# The two new rows should be gone
after_count = con.execute("SELECT count(*) FROM events").fetchone()[0]
print(f"After ROLLBACK:                     {after_count} rows  (expected: 3)")

# Confirm specific event_ids are not present
ghost_rows = con.execute("SELECT event_id FROM events WHERE event_id IN (100, 101)").fetchall()
print(f"Ghost rows (100, 101): {ghost_rows}  (expected: empty)")

**What just happened?**
- `BEGIN` started an explicit transaction — auto-commit was suspended
- The rows were visible **inside the transaction** (to the same connection)
- `ROLLBACK` undid all changes atomically — zero rows leaked to the persistent file
- **This is Atomicity** — either everything commits or nothing does

---
## Step 4 · Demonstrate COMMIT — changes survive a reconnect

A committed transaction is durable: even if the process is killed immediately after, the data is safe on disk.

In [ ]:
# Start a transaction and commit this time
con.execute("BEGIN")

con.execute("""
INSERT INTO events (event_id, event_type, payload) VALUES
    (10, 'checkout_start', '{"cart_id": 55, "user": 77}'),
    (11, 'checkout_complete', '{"cart_id": 55, "order_id": 999, "user": 77}')
""")

con.execute("COMMIT")

print("After COMMIT:", con.execute("SELECT count(*) FROM events").fetchone()[0], "rows")

# Close and reopen — committed rows must survive
con.close()
con = duckdb.connect(db_path)

rows_after_reopen = con.execute("SELECT count(*) FROM events").fetchone()[0]
print(f"After close + reopen: {rows_after_reopen} rows  (expected: 5)")
print()
print(con.execute("SELECT event_id, event_type FROM events ORDER BY event_id").df().to_string(index=False))

**What just happened?**
- `COMMIT` flushed the two new rows to the `.duckdb` file
- Closing and reopening the connection confirmed the data is permanently stored
- **This is Durability** — committed transactions survive process restarts

---
## Step 5 · ATTACH — cross-database JOINs

`ATTACH` mounts a second DuckDB file and makes its tables accessible via a namespace prefix:

```sql
ATTACH 'other.duckdb' AS other;
SELECT * FROM other.some_table;
DETACH other;
```

This lets you:
- Join across database files without copying data
- Read-only attach a production file from an analytics script
- Combine a local file with a remote S3 file (via `httpfs`)

In [ ]:
import duckdb, os

# Create a second database with user dimension data
users_db_path = '/tmp/users.duckdb'
if os.path.exists(users_db_path):
    os.remove(users_db_path)

users_con = duckdb.connect(users_db_path)
users_con.execute("""
CREATE TABLE users AS
SELECT
    i                                AS user_id,
    'user_' || i                     AS username,
    CASE i % 3
        WHEN 0 THEN 'free'
        WHEN 1 THEN 'pro'
        ELSE 'enterprise'
    END AS plan
FROM range(1, 201) t(i)
""")
users_con.close()  # close to release the write lock

print("users.duckdb created with",
      duckdb.connect(users_db_path).execute("SELECT count(*) FROM users").fetchone()[0], "rows")

# Now ATTACH the users database inside our main analytics connection
con.execute(f"ATTACH '{users_db_path}' AS userdb (READ_ONLY)")

# Cross-database JOIN: events (analytics.duckdb) × users (users.duckdb)
# The 'user' field in the JSON payload contains the user_id
result = con.execute("""
SELECT
    e.event_type,
    e.event_id,
    CAST(json_extract(e.payload, '$.user') AS INT) AS user_id,
    u.username,
    u.plan
FROM events e
JOIN userdb.users u
  ON CAST(json_extract(e.payload, '$.user') AS INT) = u.user_id
ORDER BY e.event_id
""").df()
print("\nCross-database JOIN result:")
print(result.to_string(index=False))

# Detach when done
con.execute("DETACH userdb")
print("\nuserdb detached.")

**What just happened?**
- `ATTACH '…' AS userdb (READ_ONLY)` mounted the second file without copying it
- Tables in `userdb` are accessed as `userdb.users` — a namespace prefix
- `json_extract(payload, '$.user')` pulled the user_id from the JSON payload column
- **READ_ONLY** prevents accidental writes — essential when attaching production files

---
## Step 6 · DuckDB concurrency model — single writer, multiple readers

DuckDB's concurrency model is important to understand for pipeline design:

| Scenario | What happens |
|---|---|
| Multiple read connections | Allowed concurrently |
| One write + one read | **Blocked** — write connection holds an exclusive lock |
| Two write connections | Second `connect()` raises `IOException` |

> **Reference:** [DuckDB concurrency docs](https://duckdb.org/docs/connect/concurrency)

In [ ]:
import duckdb

# Demonstrate: trying to open a second WRITE connection to the same file raises
print("con is already open for write.")
print(f"con: {con}")

try:
    # This will raise because con already holds the write lock
    con_conflict = duckdb.connect(db_path)  # second write connection
    # If it somehow succeeds, try a write to trigger the error
    con_conflict.execute("CREATE TABLE conflict_test (x INT)")
    print("UNEXPECTED: second write connection succeeded")
    con_conflict.close()
except duckdb.IOException as e:
    print(f"Expected error (write conflict): {type(e).__name__}")
    print(f"  {str(e)[:120]}")
except Exception as e:
    print(f"Got exception ({type(e).__name__}): {str(e)[:120]}")

# READ_ONLY connection to the same file — allowed alongside a write connection
try:
    con_read = duckdb.connect(db_path, read_only=True)
    count = con_read.execute("SELECT count(*) FROM events").fetchone()[0]
    print(f"\nRead-only connection succeeded: {count} events visible")
    con_read.close()
except Exception as e:
    print(f"Read-only connection failed: {e}")

**What just happened?**
- A second write connection to the same file is blocked or raises an IOException
- A `read_only=True` connection can coexist with a write connection — readers see a consistent snapshot
- **Pipeline design implication:** design pipelines so that one process writes while orchestrators and dashboards open read-only connections
- For concurrent writes, partition data into separate `.duckdb` files (one per day, entity, or tenant)

---
## Step 7 · Safe upsert with INSERT OR REPLACE

Idempotent pipelines need upserts: insert a row if it doesn't exist, replace it if it does.

DuckDB supports two forms:

```sql
-- 1. INSERT OR REPLACE — replaces the whole row on conflict
INSERT OR REPLACE INTO t VALUES (...);

-- 2. INSERT OR IGNORE — skips the row silently on conflict
INSERT OR IGNORE INTO t VALUES (...);

-- 3. INSERT … ON CONFLICT DO UPDATE SET … (standard SQL)
INSERT INTO t VALUES (...)
ON CONFLICT (id) DO UPDATE SET col = EXCLUDED.col;
```

For a pipeline that replays the same batch multiple times, `INSERT OR REPLACE` (or `ON CONFLICT DO UPDATE`) ensures repeated runs produce the same result.

In [ ]:
# Create a table with a primary key to demonstrate upserts
con.execute("""
CREATE OR REPLACE TABLE user_metrics (
    user_id       INTEGER PRIMARY KEY,
    total_events  INTEGER NOT NULL DEFAULT 0,
    last_seen     DATE,
    updated_at    TIMESTAMP DEFAULT current_timestamp
)
""")

# Initial load — first pipeline run
def run_pipeline(label: str):
    """Simulates a daily pipeline run that inserts/updates user metrics."""
    con.execute("""
    INSERT OR REPLACE INTO user_metrics (user_id, total_events, last_seen)
    VALUES
        (42,  10, '2024-01-15'),
        (77,   5, '2024-01-15'),
        (99,   3, '2024-01-15')
    """)
    rows = con.execute("SELECT user_id, total_events, last_seen FROM user_metrics ORDER BY user_id").fetchall()
    print(f"After {label}:")
    for r in rows:
        print(f"  user_id={r[0]}  events={r[1]}  last_seen={r[2]}")
    print()

run_pipeline("run 1")
run_pipeline("run 2 (same data — idempotent)")  # identical run should produce identical results

# Now simulate an update: user 42 has more events
con.execute("""
INSERT OR REPLACE INTO user_metrics (user_id, total_events, last_seen)
VALUES (42, 25, '2024-01-16')  -- updated values
""")
print("After update (user 42 events 10 → 25):")
print(con.execute("SELECT user_id, total_events, last_seen FROM user_metrics ORDER BY user_id").df().to_string(index=False))

**What just happened?**
- `INSERT OR REPLACE` replaced user 42's row on the second pipeline run — no duplicate, no error
- Running the same batch twice produced the same table state — this is **idempotency**
- When `user_id = 42` was updated, the old row was atomically replaced by the new one
- **Production pattern:** wrap the INSERT OR REPLACE in a transaction so the entire batch is atomic

---
## Step 8 · ON CONFLICT DO UPDATE — fine-grained upsert control

`INSERT OR REPLACE` replaces the entire row, which resets any columns not in the INSERT list. For partial updates (increment a counter, keep the oldest `created_at`), use `ON CONFLICT DO UPDATE SET`.

In [ ]:
# Table tracks cumulative event counts — we want to ADD new events, not replace
con.execute("""
CREATE OR REPLACE TABLE event_counts (
    user_id     INTEGER PRIMARY KEY,
    event_count INTEGER NOT NULL DEFAULT 0,
    first_seen  DATE,
    last_seen   DATE
)
""")

def ingest_batch(batch: list):
    """Ingest a batch of (user_id, events_today, date) tuples."""
    for user_id, new_events, dt in batch:
        con.execute("""
        INSERT INTO event_counts (user_id, event_count, first_seen, last_seen)
        VALUES (?, ?, ?, ?)
        ON CONFLICT (user_id) DO UPDATE SET
            event_count = event_counts.event_count + EXCLUDED.event_count,
            last_seen   = GREATEST(event_counts.last_seen, EXCLUDED.last_seen)
        """, [user_id, new_events, dt, dt])

# Day 1
ingest_batch([(42, 5, '2024-01-10'), (77, 3, '2024-01-10')])
print("After Day 1 ingest:")
print(con.execute("SELECT * FROM event_counts ORDER BY user_id").df().to_string(index=False))

# Day 2 — user 42 gets more events; user 99 is new
ingest_batch([(42, 8, '2024-01-11'), (99, 12, '2024-01-11')])
print("\nAfter Day 2 ingest:")
print(con.execute("SELECT * FROM event_counts ORDER BY user_id").df().to_string(index=False))

# Replay Day 2 again (idempotency test) — user 42 should NOT get double-counted
# (This only works properly if the pipeline uses deduplication before ingesting.)
print("\nReplaying Day 2 (same events) would double-count — see Note below.")
print("For true idempotency, track processed batch IDs or use INSERT OR IGNORE.")

**What just happened?**
- `ON CONFLICT … DO UPDATE SET event_count = event_counts.event_count + EXCLUDED.event_count` increments the counter rather than replacing it
- `EXCLUDED` refers to the row that *would have been inserted* — it's the source of truth for the update expression
- `GREATEST(...)` keeps the most-recent `last_seen` date without comparing in application code
- **Important:** accumulating upserts are not naturally idempotent — track batch IDs to prevent double-counting on replay

---
## Challenge — wrap a multi-table upsert in a transaction

Combine transactions and upserts from today's notebook.

In [ ]:
# Challenge:
# You receive a batch of order updates. Some order_ids already exist;
# others are new. Write a function `process_order_batch(con, batch)`
# that:
#
# 1. Creates (or reuses) an orders table with columns:
#    order_id (PK), customer_id, status, total, updated_at
#
# 2. Wraps the entire batch in a transaction
#
# 3. Uses INSERT OR REPLACE to upsert each order
#
# 4. If any individual insert raises an exception, rolls back the whole batch
#
# 5. Returns (committed_count, error_message) as a tuple
#
# Test it with:
#   batch = [
#     (1001, 42, 'delivered', 99.99),
#     (1002, 77, 'pending',   45.00),
#     (1001, 42, 'refunded',  99.99),   # duplicate — should replace
#   ]

# Your solution here


---
## Day 9 key concepts recap

| Concept | What to remember |
|---|---|
| `duckdb.connect('file.duckdb')` | Creates/opens a persistent file; data survives restarts |
| `BEGIN` / `COMMIT` / `ROLLBACK` | Explicit transactions for multi-statement atomicity |
| ACID: Atomicity | All-or-nothing — ROLLBACK undoes every statement in the transaction |
| ACID: Durability | COMMIT guarantees data is on disk |
| `ATTACH 'other.duckdb' AS db` | Mount a second DuckDB file — cross-database JOINs |
| `READ_ONLY` attachment | Prevents accidental writes to production files |
| Single-writer model | Only one write connection at a time per file |
| `INSERT OR REPLACE` | Replace the whole row on PK conflict |
| `ON CONFLICT DO UPDATE SET` | Partial upsert — increment counters, keep oldest dates |

> **Tip:** DuckDB supports only one writer at a time — design your pipelines so that a single process writes while readers operate on snapshots. For concurrent writes, partition your data by day or entity and use separate files.

---
## What's next
**Day 10** → Capstone: build a full analytical lakehouse pipeline on public remote Parquet data, with window functions, EXPLAIN ANALYZE, and a Python orchestration script.

Mark Day 9 complete in your [tracker](../index.html).